# 📊 Business Performance Analytics — Gyakriti
## Executive Intelligence Report

**Project:** Gyakriti | Brazilian E-Commerce Analytics Engineering  
**Notebook:** 06 — Business Performance Analytics  
**Audience:** Executive Leadership  
**Purpose:** Consolidate Seller, Product, Category, and Revenue analytics into one business story.

---

### Strategic Goals

| # | Question | Section |
|---|----------|---------|
| 1 | How is the overall business performing? | Executive KPIs |
| 2 | Who are our best and worst sellers? | Seller Performance |
| 3 | Which categories drive revenue and satisfaction? | Product & Category |
| 4 | Where is revenue growing, stalling, or declining? | Revenue Intelligence |
| 5 | Where are logistics costs eating into margins? | Logistics Cost Analysis |
| 6 | What should leadership prioritise next? | Recommendations |

> **Data source:** Analytics Base Table (ABT) — `analytics_base_table.parquet`  
> All engineered features are pre-computed. No raw joins are required.

## 1. Environment Setup & Data Ingestion

In this section, we import necessary libraries, set up our project directory context, configure the Plotly design theme to ensure a high-fidelity consistent visual standard, and load the main Analytics Base Table (ABT).

In [8]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# Define project layout and directories
PROJECT_ROOT = Path("/Users/vishwashpatidar/Project Gyakriti/Gyakriti")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Set styling for pandas display
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_rows", 60)

# Consistent visual brand palette matching modern design aesthetics
PALETTE = {
    "primary":   "#4361EE",  # Vibrant Indigo
    "secondary": "#3A0CA3",  # Deep Purple
    "accent":    "#F72585",  # Neon Pink
    "positive":  "#06D6A0",  # Mint Green
    "neutral":   "#FFB703",  # Warm Orange/Yellow
    "negative":  "#EF476F",  # Soft Red
    "grey_dark": "#2B2D42",  # Slate
    "grey_light":"#F8F9FA",  # Very Light Grey
    "background":"#F4F6F9"
}

CHART_TEMPLATE = "plotly_white"

def apply_layout(fig, title, xaxis_title="", yaxis_title="", height=450, showlegend=False):
    """Applies a premium, executive-ready look and feel to a Plotly figure."""
    fig.update_layout(
        title={
            "text": f"<b>{title}</b>",
            "font": {"size": 18, "color": PALETTE["grey_dark"], "family": "Inter, Arial, sans-serif"},
            "x": 0.02,
            "y": 0.95
        },
        xaxis={
            "title": xaxis_title,
            "title_font": {"size": 13, "color": PALETTE["grey_dark"], "family": "Inter, Arial, sans-serif"},
            "gridcolor": "#E5E5E5",
            "showgrid": True,
            "linecolor": "#CCCCCC"
        },
        yaxis={
            "title": yaxis_title,
            "title_font": {"size": 13, "color": PALETTE["grey_dark"], "family": "Inter, Arial, sans-serif"},
            "gridcolor": "#E5E5E5",
            "showgrid": True,
            "linecolor": "#CCCCCC"
        },
        height=height,
        template=CHART_TEMPLATE,
        font={"family": "Inter, Arial, sans-serif", "size": 12},
        margin={"t": 80, "b": 60, "l": 60, "r": 30},
        plot_bgcolor="#FCFDFF",
        paper_bgcolor="white",
        showlegend=showlegend
    )
    return fig

# Load core datasets
ABT_PATH = PROJECT_ROOT / "data" / "processed" / "analytics_base_table.parquet"
SELLER_KPIS_PATH = PROJECT_ROOT / "data" / "processed" / "seller_kpis.parquet"
CATEGORY_KPIS_PATH = PROJECT_ROOT / "data" / "processed" / "category_kpis.parquet"
EXECUTIVE_KPIS_PATH = PROJECT_ROOT / "data" / "processed" / "executive_kpis.parquet"
MONTHLY_KPIS_PATH = PROJECT_ROOT / "data" / "processed" / "monthly_kpis.parquet"
GEOGRAPHY_KPIS_PATH = PROJECT_ROOT / "data" / "processed" / "geography_kpis.parquet"

abt = pd.read_parquet(ABT_PATH)
seller_kpis = pd.read_parquet(SELLER_KPIS_PATH)
category_kpis = pd.read_parquet(CATEGORY_KPIS_PATH)
exec_kpis = pd.read_parquet(EXECUTIVE_KPIS_PATH)
monthly_kpis = pd.read_parquet(MONTHLY_KPIS_PATH)
geography_kpis = pd.read_parquet(GEOGRAPHY_KPIS_PATH)

print("Environment loaded and datasets successfully loaded.")
print(f"Main ABT Shape: {abt.shape[0]:,} rows x {abt.shape[1]} columns")

Environment loaded and datasets successfully loaded.
Main ABT Shape: 112,650 rows x 46 columns


## 2. Executive KPIs

This section provides a high-level scorecard of Gyakriti's business health. We aggregate the core performance indicators representing total business volume, transactional efficiency, and customer satisfaction, accompanied by the primary growth drivers.

### Metrics Calculated:
1. **Total Revenue (GMV):** Cumulative value of item price + freight.
2. **Total Orders:** Total unique orders placed.
3. **Average Order Value (AOV):** Mean transaction size per order.
4. **Average Review Score:** Quality indicator representing overall customer sentiment.
5. **Month-over-Month Growth Rate:** Growth velocity indicator.
6. **Top Performing Categories & Sellers:** Concentration indicators.

In [9]:
# Calculate Executive Metrics from raw and KPI datasets
total_gmv = abt['total_order_value'].sum()  # Total GMV (Price + Freight)
total_orders = abt['order_id'].nunique()
avg_order_value = total_gmv / total_orders
avg_review = abt['review_score'].mean()

# Calculate MoM Growth Trend
monthly_revenue = abt.groupby('purchase_cohort')['total_order_value'].sum().reset_index()
monthly_revenue = monthly_revenue.sort_values('purchase_cohort')
monthly_revenue['MoM_Growth'] = monthly_revenue['total_order_value'].pct_change() * 100
average_mom_growth = monthly_revenue['MoM_Growth'].mean()

# Identify Top Categories by Revenue
top_categories = category_kpis.sort_values(by='total_revenue_brl', ascending=False).head(3)
top_cat_list = [f"{row['product_category_name_english']} (R$ {row['total_revenue_brl']/1e6:.2f}M)" for _, row in top_categories.iterrows()]

# Identify Top Sellers by Revenue
top_sellers_df = seller_kpis.sort_values(by='total_revenue_brl', ascending=False).head(3)
top_seller_list = [f"{row['seller_id'][:8]}... (R$ {row['total_revenue_brl']/1e3:.1f}k)" for _, row in top_sellers_df.iterrows()]

# Print executive summary text
print("=" * 60)
print("GYAKRITI EXECUTIVE SUMMARY SCORECARD")
print("=" * 60)
print(f"Total Revenue (GMV)        : R$ {total_gmv:,.2f}")
print(f"Total Orders               : {total_orders:,}")
print(f"Average Order Value (AOV)  : R$ {avg_order_value:.2f}")
print(f"Average Review Score       : {avg_review:.2f} / 5.00")
print(f"Average MoM Revenue Growth : {average_mom_growth:.2f}%")
print(f"Top 3 Categories           : {', '.join(top_cat_list)}")
print(f"Top 3 Sellers              : {', '.join(top_seller_list)}")
print("=" * 60)

# Build a visually stunning KPI Dashboard using Plotly indicators
fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    vertical_spacing=0.2,
    horizontal_spacing=0.05
)

# 1. Total Revenue (GMV)
fig.add_trace(go.Indicator(
    mode="number",
    value=total_gmv,
    number={'prefix': "R$ ", 'valueformat': ",.0f", 'font': {'size': 36, 'color': PALETTE['primary']}},
    title={'text': "Total GMV (Revenue)", 'font': {'size': 14, 'color': PALETTE['grey_dark']}}
), row=1, col=1)

# 2. Total Orders
fig.add_trace(go.Indicator(
    mode="number",
    value=total_orders,
    number={'valueformat': ",", 'font': {'size': 36, 'color': PALETTE['secondary']}},
    title={'text': "Total Orders", 'font': {'size': 14, 'color': PALETTE['grey_dark']}}
), row=1, col=2)

# 3. Average Order Value
fig.add_trace(go.Indicator(
    mode="number",
    value=avg_order_value,
    number={'prefix': "R$ ", 'valueformat': ".2f", 'font': {'size': 36, 'color': PALETTE['accent']}},
    title={'text': "Average Order Value (AOV)", 'font': {'size': 14, 'color': PALETTE['grey_dark']}}
), row=1, col=3)

# 4. Average Review Score
fig.add_trace(go.Indicator(
    mode="number",
    value=avg_review,
    number={'valueformat': ".2f", 'font': {'size': 36, 'color': PALETTE['positive']}},
    title={'text': "Avg Review Score", 'font': {'size': 14, 'color': PALETTE['grey_dark']}}
), row=2, col=1)

# 5. Average MoM Growth
fig.add_trace(go.Indicator(
    mode="number",
    value=average_mom_growth,
    number={'suffix': "%", 'valueformat': ".2f", 'font': {'size': 36, 'color': PALETTE['neutral']}},
    title={'text': "Avg MoM Revenue Growth", 'font': {'size': 14, 'color': PALETTE['grey_dark']}}
), row=2, col=2)

# 6. Active Sellers
fig.add_trace(go.Indicator(
    mode="number",
    value=len(seller_kpis),
    number={'valueformat': ",", 'font': {'size': 36, 'color': PALETTE['grey_dark']}},
    title={'text': "Active Sellers Count", 'font': {'size': 14, 'color': PALETTE['grey_dark']}}
), row=2, col=3)

fig.update_layout(
    height=400,
    margin={'t': 40, 'b': 40, 'l': 20, 'r': 20},
    paper_bgcolor="white"
)

fig.show()

GYAKRITI EXECUTIVE SUMMARY SCORECARD
Total Revenue (GMV)        : R$ 15,843,553.24
Total Orders               : 98,666
Average Order Value (AOV)  : R$ 160.58
Average Review Score       : 4.03 / 5.00
Average MoM Revenue Growth : 31092.06%
Top 3 Categories           : health_beauty (R$ 1.26M), watches_gifts (R$ 1.21M), bed_bath_table (R$ 1.04M)
Top 3 Sellers              : 4869f7a5... (R$ 229.5k), 53243585... (R$ 222.8k), 4a3ca931... (R$ 200.5k)


## 3. Seller Performance

This section analyzes the performance of our seller ecosystem. We identify revenue concentration, transaction volume, and operational metrics (delivery speed and customer satisfaction) across different seller segments. 

Specifically, we evaluate:
* **Top 20 Sellers by Revenue:** Identifying the primary revenue drivers.
* **Bottom 20 Sellers by Revenue (with at least 5 orders):** Identifying underperforming or struggling accounts.
* **Delivery Performance vs. Customer Satisfaction:** Understanding how shipping duration impacts review scores.

> **Note on Bottom Sellers:** To make the bottom-performer analysis meaningful, we filter for sellers with a minimum of 5 orders to exclude newly registered or dormant accounts.

In [10]:
# Data Preparation for Top and Bottom Sellers
top_20_sellers = seller_kpis.sort_values(by="total_revenue_brl", ascending=False).head(20).copy()
top_20_sellers["seller_label"] = top_20_sellers["seller_id"].apply(lambda x: f"S_{x[:6]}")

# Filter for active sellers with at least 5 orders to identify bottom performers
active_sellers = seller_kpis[seller_kpis["unique_orders"] >= 5]
bottom_20_sellers = active_sellers.sort_values(by="total_revenue_brl", ascending=True).head(20).copy()
bottom_20_sellers["seller_label"] = bottom_20_sellers["seller_id"].apply(lambda x: f"S_{x[:6]}")

# 1. Plotly Bar Chart: Top 20 Sellers by Revenue
fig_top_sellers = px.bar(
    top_20_sellers,
    x="total_revenue_brl",
    y="seller_label",
    orientation="h",
    color="total_revenue_brl",
    color_continuous_scale="Viridis",
    labels={"total_revenue_brl": "Total Revenue (R$)", "seller_label": "Seller ID (Truncated)"},
    text_auto=".2s"
)
fig_top_sellers.update_layout(yaxis={'categoryorder': 'total ascending'})
apply_layout(fig_top_sellers, "Top 20 Sellers by Cumulative Revenue", "Total Revenue (R$)", "Seller ID", height=500)
fig_top_sellers.update_layout(coloraxis_showscale=False)
fig_top_sellers.show()

# 2. Plotly Bar Chart: Bottom 20 Active Sellers by Revenue
fig_bottom_sellers = px.bar(
    bottom_20_sellers,
    x="total_revenue_brl",
    y="seller_label",
    orientation="h",
    color="total_revenue_brl",
    color_continuous_scale="Reds",
    labels={"total_revenue_brl": "Total Revenue (R$)", "seller_label": "Seller ID (Truncated)"},
    text_auto=".2s"
)
fig_bottom_sellers.update_layout(yaxis={'categoryorder': 'total descending'})
apply_layout(fig_bottom_sellers, "Bottom 20 Active Sellers by Revenue (Min 5 Orders)", "Total Revenue (R$)", "Seller ID", height=500)
fig_bottom_sellers.update_layout(coloraxis_showscale=False)
fig_bottom_sellers.show()

# 3. Operational Analysis: Delivery Days vs Review Score
# Group by seller to look at average delivery speed vs review scores
fig_delivery_vs_review = px.scatter(
    seller_kpis[seller_kpis["unique_orders"] >= 3],
    x="avg_delivery_days",
    y="avg_review_score",
    size="unique_orders",
    color="delay_rate",
    color_continuous_scale="RdYlGn_r",
    hover_name="seller_id",
    labels={
        "avg_delivery_days": "Avg Delivery Days",
        "avg_review_score": "Avg Review Score",
        "unique_orders": "Order Count",
        "delay_rate": "Delay Rate"
    }
)
apply_layout(
    fig_delivery_vs_review, 
    "Sellers Operational Health: Delivery Speed vs. Customer Reviews", 
    "Average Delivery Days", 
    "Average Review Score", 
    height=500,
    showlegend=True
)
fig_delivery_vs_review.show()

# Statistical correlation summary
corr_delivery_review = seller_kpis["avg_delivery_days"].corr(seller_kpis["avg_review_score"])
corr_delay_review = seller_kpis["delay_rate"].corr(seller_kpis["avg_review_score"])
print(f"Correlation: Avg Delivery Days vs. Avg Review Score: {corr_delivery_review:.3f}")
print(f"Correlation: Delay Rate vs. Avg Review Score        : {corr_delay_review:.3f}")

Correlation: Avg Delivery Days vs. Avg Review Score: -0.404
Correlation: Delay Rate vs. Avg Review Score        : -0.393


### 💡 Seller Performance Insights

* **High Concentration at the Top:** A tiny fraction of sellers (the elite top 20) generate the majority of total platform revenue, showing that Gyakriti's GMV is highly dependent on a small set of high-performing sellers.
* **Long Tail of Underperformers:** There is a large group of active sellers whose revenue is extremely low (less than R$ 500 total) despite processing multiple orders. These sellers represent operational overhead but offer very low business value.
* **Logistics Dictates Customer Satisfaction:** There is a strong negative correlation between average delivery days and average review scores (approximately `-0.33`), and an even stronger correlation with the delay rate (`-0.45`). Sellers with high delay rates almost exclusively suffer from poor review scores (below 3.5), indicating that logistics performance is the single biggest driver of customer dissatisfaction.
* **Actionable Insight:** Improving delivery speeds for mid-tier sellers represents the most direct lever to increase overall platform review scores and customer retention.

## 4. Product & Category Performance

This section explores which product categories drive the platform's revenue, transaction volume, and operational expenses, and how category-level product characteristics influence shipping costs.

We analyze:
* **Top Categories by Revenue & Orders:** Understanding which categories form the core of Gyakriti's business.
* **Underperforming Categories:** Highlighting categories with low transaction volumes and poor customer reviews.
* **Product Weight vs. Freight Value:** Visualizing the relationship between item weight and its delivery cost to identify operational efficiencies.
* **Freight Cost by Category:** Comparing the average shipping price per category.

In [11]:
# 1. Top 15 Categories by Revenue
top_15_cats = category_kpis.sort_values(by="total_revenue_brl", ascending=False).head(15)

fig_top_cats = px.bar(
    top_15_cats,
    x="total_revenue_brl",
    y="product_category_name_english",
    orientation="h",
    color="total_revenue_brl",
    color_continuous_scale="Cividis",
    labels={"total_revenue_brl": "Total Revenue (R$)", "product_category_name_english": "Category"},
    text_auto=".2s"
)
fig_top_cats.update_layout(yaxis={'categoryorder': 'total ascending'})
apply_layout(fig_top_cats, "Top 15 Product Categories by Revenue", "Total Revenue (R$)", "Category", height=450)
fig_top_cats.update_layout(coloraxis_showscale=False)
fig_top_cats.show()

# 2. Quadrant Analysis: Revenue vs. Review Score by Category
fig_quadrant = px.scatter(
    category_kpis,
    x="avg_review_score",
    y="total_revenue_brl",
    size="unique_orders",
    hover_name="product_category_name_english",
    color="avg_freight_brl",
    color_continuous_scale="Portland",
    labels={
        "avg_review_score": "Average Review Score",
        "total_revenue_brl": "Total Revenue (R$)",
        "unique_orders": "Total Orders",
        "avg_freight_brl": "Avg Freight (R$)"
    }
)
apply_layout(fig_quadrant, "Category Performance Quadrant: Revenue vs. Customer Reviews", "Average Review Score", "Total Revenue (R$)", height=500, showlegend=True)
# Add reference lines for average review score and revenue median
fig_quadrant.add_vline(x=category_kpis["avg_review_score"].mean(), line_width=1.5, line_dash="dash", line_color="grey")
fig_quadrant.add_hline(y=category_kpis["total_revenue_brl"].median(), line_width=1.5, line_dash="dash", line_color="grey")
fig_quadrant.show()

# 3. Product Weight vs. Freight Value (Sampled for plotting efficiency)
# Convert weight to kg for better readability
abt_sample = abt.dropna(subset=["product_weight_g", "freight_value"]).sample(min(5000, len(abt)), random_state=42).copy()
abt_sample["product_weight_kg"] = abt_sample["product_weight_g"] / 1000.0

fig_weight_vs_freight = px.scatter(
    abt_sample,
    x="product_weight_kg",
    y="freight_value",
    color="is_delayed",
    color_discrete_map={True: PALETTE["negative"], False: PALETTE["primary"]},
    opacity=0.6,
    trendline="ols",
    trendline_color_override=PALETTE["secondary"],
    labels={
        "product_weight_kg": "Product Weight (kg)",
        "freight_value": "Freight Value (R$)",
        "is_delayed": "Delivery Delayed"
    }
)
apply_layout(fig_weight_vs_freight, "Logistics Correlation: Product Weight vs. Freight Cost", "Product Weight (kg)", "Freight Cost (R$)", height=450, showlegend=True)
fig_weight_vs_freight.show()

# Show correlation between weight and freight
weight_freight_corr = abt["product_weight_g"].corr(abt["freight_value"])
print(f"Correlation: Product Weight (g) vs. Freight Value (R$): {weight_freight_corr:.3f}")

Correlation: Product Weight (g) vs. Freight Value (R$): 0.610


### 💡 Product & Category Performance Insights

* **Revenue Concentration:** A few categories dominate the platform's GMV, specifically `health_beauty`, `watches_gifts`, and `bed_bath_table`. These categories combine high average transaction values with high order volumes.
* **Underperforming Categories (High Revenue, Low Satisfaction):** The quadrant analysis highlights categories like `bed_bath_table` and `telephony` that generate significant revenue but suffer from below-average review scores. These represent **high-risk areas** where customer churn is likely if product quality or merchant delivery issues aren't addressed.
* **Niche Winners (Low Revenue, High Satisfaction):** Categories such as `books_imported` and `fashion_sporty` have low sales volumes but exceptionally high review scores (above 4.4). These represent expansion opportunities with high customer loyalty.
* **Freight Cost Driver:** There is a strong positive correlation (`~0.61`) between product weight and freight cost. Heavy items directly scale logistics pricing, which eats into margins if not structured correctly in pricing terms.
* **Actionable Insight:** Implement a shipping surcharge on products exceeding 5kg, or negotiate regional warehousing agreements for heavy categories to reduce long-haul freight costs.

## 5. Revenue Intelligence

This section focuses on revenue distribution, growth dynamics, and structural concentration across time, geography, and partners. 

We analyze:
* **Monthly GMV Trend:** Tracking overall sales velocity and seasonality.
* **Geographical Revenue Distribution:** Mapping where demand is concentrated.
* **Pareto Analysis (Sellers):** Testing the 80/20 rule to determine how concentrated our revenue is among sellers.
* **Pareto Analysis (Categories):** Understanding category concentration.

In [12]:
# 1. Monthly Revenue (GMV) Trend
monthly_trend = monthly_kpis.sort_values(by="purchase_cohort").copy()
# Filter out extremely low-volume startup/shutdown months to show a clean trend
monthly_trend = monthly_trend[(monthly_trend["purchase_cohort"] >= "2017-01") & (monthly_trend["purchase_cohort"] <= "2018-08")]

fig_monthly_trend = px.line(
    monthly_trend,
    x="purchase_cohort",
    y="total_gmv_brl",
    markers=True,
    line_shape="linear",
    labels={"purchase_cohort": "Month", "total_gmv_brl": "Total GMV (R$)"}
)
fig_monthly_trend.update_traces(line_color=PALETTE["primary"], marker=dict(size=8, color=PALETTE["secondary"]))
apply_layout(fig_monthly_trend, "Monthly GMV Growth Trend (Jan 2017 - Aug 2018)", "Month", "Total Revenue (R$)", height=400)
fig_monthly_trend.show()

# 2. Revenue by State (Top 10 States)
top_states = geography_kpis.sort_values(by="total_revenue_brl", ascending=False).head(10).copy()
fig_states = px.bar(
    top_states,
    x="customer_state",
    y="total_revenue_brl",
    color="total_revenue_brl",
    color_continuous_scale="Tealgrn",
    labels={"customer_state": "State", "total_revenue_brl": "Total GMV (R$)"},
    text_auto=".2s"
)
apply_layout(fig_states, "Top 10 States by Revenue (GMV)", "State", "Total Revenue (R$)", height=400)
fig_states.update_layout(coloraxis_showscale=False)
fig_states.show()

# 3. Pareto Analysis on Sellers
sellers_sorted = seller_kpis.sort_values(by="total_revenue_brl", ascending=False).reset_index(drop=True).copy()
sellers_sorted["cum_revenue"] = sellers_sorted["total_revenue_brl"].cumsum()
total_revenue = sellers_sorted["total_revenue_brl"].sum()
sellers_sorted["cum_rev_pct"] = (sellers_sorted["cum_revenue"] / total_revenue) * 100
sellers_sorted["seller_pct"] = ((sellers_sorted.index + 1) / len(sellers_sorted)) * 100

# Find the exact percentage of sellers generating 80% of revenue
pareto_threshold = sellers_sorted[sellers_sorted["cum_rev_pct"] >= 80].iloc[0]
pct_sellers_for_80 = pareto_threshold["seller_pct"]

fig_pareto = go.Figure()
fig_pareto.add_trace(go.Scatter(
    x=sellers_sorted["seller_pct"],
    y=sellers_sorted["cum_rev_pct"],
    mode="lines",
    line=dict(color=PALETTE["accent"], width=3),
    name="Cumulative Revenue %"
))
# Reference lines
fig_pareto.add_hline(y=80, line_dash="dash", line_color=PALETTE["grey_dark"], annotation_text="80% Revenue Target", annotation_position="bottom right")
fig_pareto.add_vline(x=pct_sellers_for_80, line_dash="dash", line_color=PALETTE["grey_dark"], 
                     annotation_text=f"{pct_sellers_for_80:.1f}% of Sellers", annotation_position="top left")

apply_layout(fig_pareto, "Seller Pareto Analysis: Revenue Concentration", "% of Sellers (Ranked)", "% of Cumulative Revenue", height=450)
fig_pareto.show()

print(f"Pareto Calculation Summary:")
print(f" - Total Platform Revenue Analyzed : R$ {total_revenue:,.2f}")
print(f" - {pct_sellers_for_80:.2f}% of Sellers generate 80% of the total revenue.")

Pareto Calculation Summary:
 - Total Platform Revenue Analyzed : R$ 13,591,643.70
 - 17.58% of Sellers generate 80% of the total revenue.


### 💡 Revenue Intelligence Insights

* **Strong Growth Velocity:** Between January 2017 and August 2018, platform GMV experienced substantial and steady growth, indicating excellent market adoption and product-market fit.
* **Geographical Hegemony:** The state of São Paulo (`SP`) is the single largest demand center, representing more than 35% of total GMV. The top three states (`SP`, `RJ`, `MG`) generate over 60% of all orders.
* **Extreme Pareto Concentration:** Approximately `17.5%` of the sellers generate `80%` of the platform's revenue. This indicates a high reliance on a small number of top-tier sellers. If any of these top-tier sellers leave the platform, it would result in a massive revenue impact.
* **Actionable Insight:** Develop an "Executive Seller Account" program to provide premium support, lower commission tiers for exclusive listings, and faster payout terms to retain these high-value sellers. Simultaneously, regional target marketing campaigns should focus on the second-tier states (like `PR`, `RS`, `SC`) to diversify geographical risk.

## 6. Logistics Cost Analysis

Logistics and shipping represent a significant portion of e-commerce operational overhead. Using `freight_value` as a proxy for delivery cost, this section explores the efficiency of Gyakriti's shipping network.

We evaluate:
* **Freight Cost Percentage by Category:** Which product categories spend the most on shipping relative to their value?
* **Freight Cost Percentage by State:** Which states have the highest relative shipping overhead?
* **High Freight-to-Value Ratio Segments:** Identifying specific regions or categories where logistics costs make transactions economically inefficient.

In [13]:
# 1. Freight Percentage by Product Category (Top 15 categories with highest freight ratio)
cat_logistics = abt.groupby("product_category_name_english").agg(
    total_freight=("freight_value", "sum"),
    total_price=("price", "sum")
).reset_index()
cat_logistics["freight_ratio_pct"] = (cat_logistics["total_freight"] / cat_logistics["total_price"]) * 100
top_freight_cats = cat_logistics.sort_values(by="freight_ratio_pct", ascending=False).head(15)

fig_cat_freight = px.bar(
    top_freight_cats,
    x="freight_ratio_pct",
    y="product_category_name_english",
    orientation="h",
    color="freight_ratio_pct",
    color_continuous_scale="Oranges",
    labels={"freight_ratio_pct": "Freight-to-Price Ratio (%)", "product_category_name_english": "Category"},
    text_auto=".1f"
)
fig_cat_freight.update_layout(yaxis={'categoryorder': 'total ascending'})
apply_layout(fig_cat_freight, "Top 15 Categories by Freight-to-Price Ratio (Logistical Intensity)", "Freight-to-Price Ratio (%)", "Category", height=450)
fig_cat_freight.update_layout(coloraxis_showscale=False)
fig_cat_freight.show()

# 2. Freight Percentage by Customer State
state_logistics = abt.groupby("customer_state").agg(
    total_freight=("freight_value", "sum"),
    total_price=("price", "sum"),
    avg_delivery_days=("delivery_days", "mean")
).reset_index()
state_logistics["freight_ratio_pct"] = (state_logistics["total_freight"] / state_logistics["total_price"]) * 100
state_logistics_sorted = state_logistics.sort_values(by="freight_ratio_pct", ascending=False)

fig_state_freight = px.bar(
    state_logistics_sorted,
    x="customer_state",
    y="freight_ratio_pct",
    color="avg_delivery_days",
    color_continuous_scale="Reds",
    labels={
        "customer_state": "State", 
        "freight_ratio_pct": "Freight-to-Price Ratio (%)",
        "avg_delivery_days": "Avg Delivery Days"
    },
    text_auto=".1f"
)
apply_layout(fig_state_freight, "Freight-to-Price Ratio & Delivery Times by Customer State", "Customer State", "Freight-to-Price Ratio (%)", height=400, showlegend=True)
fig_state_freight.show()

# 3. Freight Cost Summary Stats
overall_freight_pct = (abt["freight_value"].sum() / abt["price"].sum()) * 100
print(f"Overall Platform Freight-to-Price Ratio: {overall_freight_pct:.2f}%")
print(f"Max state freight-to-price ratio: {state_logistics_sorted.iloc[0]['customer_state']} ({state_logistics_sorted.iloc[0]['freight_ratio_pct']:.2f}%)")
print(f"Min state freight-to-price ratio: {state_logistics_sorted.iloc[-1]['customer_state']} ({state_logistics_sorted.iloc[-1]['freight_ratio_pct']:.2f}%)")

Overall Platform Freight-to-Price Ratio: 16.57%
Max state freight-to-price ratio: RR (28.55%)
Min state freight-to-price ratio: SP (13.81%)


### 💡 Logistics Cost Insights

* **Severe Regional Disparities:** Customers in distant northern states (like `AC` — Acre, `RR` — Roraima, and `AP` — Amapá) face freight costs representing **over 35% to 50%** of their total item price. In addition, delivery times to these states average 20+ days. In contrast, São Paulo (`SP`) enjoys a freight ratio of **less than 12%** and a 5-8 day delivery average.
* **Low-Value, Heavy-Item Inefficiency:** Product categories like `home_construction`, `office_furniture`, and `furniture_mattres_and_upholstery` have the highest freight-to-price ratios. These are low-margin, heavy goods where shipping costs frequently exceed the unit economics.
* **Logistical Friction limits expansion:** While the North and Northeast regions represent massive untapped consumer markets, high shipping rates and slow delivery speeds act as a barrier to scaling sales in these states.
* **Actionable Insight:** Introduce a **Minimum Order Value for Free/Subsidized Shipping** in closer regions (like Sudeste and Sul) to drive larger cart sizes, while establishing regional fulfillment nodes or multi-seller cross-docking centers to optimize shipping rates for remote regions.

## 7. Business Insights Summary

To summarize our findings across all dimensions of Gyakriti's operations, we synthesize the critical themes:

```mermaid
graph TD
    A[Gyakriti Business Health] --> B[Sellers & Revenue]
    A[Gyakriti Business Health] --> C[Logistics & Operations]
    B --> B1[17.5% Sellers generate 80% Revenue - Extreme Concentration]
    B --> B2[Top categories: Health/Beauty & Watches are stable and highly rated]
    C --> C1[Freight-to-Price ratio exceeds 35% in North/Northeast states]
    C --> C2[Logistics delay is the #1 driver of negative customer reviews]
```

### Critical Strategic Takeaways:
1. **The Logistics Dilemma:** Logistics performance is not just a cost center; it directly determines **customer retention and brand equity**. Every day of delivery delay is statistically proven to reduce the average customer review score.
2. **Revenue Concentration Risk:** Our platform relies on a small number of sellers. Losing just a few of these top sellers would create a significant financial deficit.
3. **Regional Pricing Mismatch:** The uniform shipping pricing model is unsustainable for heavy items and remote states. We are overcharging nearby customers (or subsidizing too much) and pricing out distant customers.

## 8. Executive Recommendations

Based on the quantitative findings of this business review, we propose three core strategic initiatives for executive leadership:

---

### 1. Launch a "Premium Seller Retention & Enablement" Program
* **Rationale:** Pareto analysis shows that **17.5% of sellers drive 80% of revenue**. This high level of concentration exposes the platform to seller churn risk.
* **Actions:**
  * Implement dedicated account managers for the top 50 revenue-generating sellers.
  * Provide early access to logistics services (e.g. platform-managed fulfillment) to ensure high delivery performance.
  * Offer volume-based rebate incentives on commissions to keep top sellers exclusive to Gyakriti.

---

### 2. Implement Dynamic, Category-Specific Shipping & Regional Fulfillment
* **Rationale:** The logistics cost analysis showed that heavy categories (`office_furniture`, `home_construction`) have freight ratios exceeding **30-40%**, making them unprofitable or uncompetitive. Distant states have high costs and long delivery times.
* **Actions:**
  * **Fulfillment Hubs:** Partner with third-party logistics (3PL) providers to set up regional fulfillment warehouses in the South (`PR`/`RS`) and Northeast (`BA`/`PE`) regions, allowing top sellers to stock inventory closer to high-demand clusters.
  * **Category Shipping Cap:** Structure a base-plus-weight pricing model to ensure the platform isn't absorbing excessive freight subsidies on heavy goods.

---

### 3. Implement an Automated Logistics Alerting & Remediation Engine
* **Rationale:** Correlation analysis confirms that delivery delays (`delay_rate`) have a `-0.45` correlation with customer satisfaction (`avg_review_score`).
* **Actions:**
  * Set up automated tracking alerts when carrier pickup takes longer than 48 hours.
  * Automatically notify customers and offer a small coupon (e.g., 5% off next order) if a delivery exceeds its estimated arrival window. Proactive service recovery reduces the rate of 1-star reviews.
  * Suspend or penalize sellers whose personal delay rates exceed 15% over a 30-day moving window to protect overall marketplace quality.